# Batch processing through your ASE Presidio APIs

Analyze and anonymize lists, Pandas tables, and nested JSON using the existing analyzer and anonymizer REST services. No local Presidio engines, spaCy models, multiprocessing, or process monitoring are needed. Processing runs on the services; this kernel organizes requests and restores the input structure.

Run on your Azure VM/kernel with private DNS and HTTPS access to both services, using the same managed identity as the working [ASE connection notebook](presidio_ase_notebook.ipynb). Configure both registered API resources and the calling user-assigned identity's **client ID**, not its principal/object ID. Values in another notebook are not automatically available here.

For your working Government v1-token configuration, App Service's issuer URL is `https://login.microsoftonline.us/<tenant-id>/`, without `/v2.0`. Keep the tenant and caller restrictions on **both** services. Keep TLS verification enabled; use `REQUESTS_CA_BUNDLE` if an organizational CA is required.

Install on the **kernel host**, using the existing environment (Windows example):

```powershell
uv pip install --python .venv-ase\Scripts\python.exe requests azure-identity ipykernel pandas
```

Select that environment as the kernel. Use Python 3.10 through 3.14 to stay in the repository's supported range. This notebook does not change Azure resources or permissions.

**Privacy:** samples are synthetic. Never display real input, tokens, raw error bodies, or whole outputs (missed detections and deliberately skipped fields remain unchanged). Only counts and shapes are printed. Clear outputs before sharing. Detection does not guarantee that all PII has been found.


In [ ]:
import copy
import math
import os
from urllib.parse import urlsplit

import pandas as pd
import requests
from azure.core.exceptions import AzureError
from azure.identity import ManagedIdentityCredential

# Set these to the exact resource values from your working ASE notebook.
ANALYZER_API_RESOURCE = os.environ.get("ANALYZER_API_RESOURCE", "").strip()
ANONYMIZER_API_RESOURCE = os.environ.get("ANONYMIZER_API_RESOURCE", "").strip()
MANAGED_IDENTITY_CLIENT_ID = os.environ.get("MANAGED_IDENTITY_CLIENT_ID", "").strip()
SERVICES = {
    "analyzer": {
        "url": os.environ.get(
            "ANALYZER_BASE_URL", "https://presidio-analyzer.ase-jtl.appserviceenvironment.us"
        ).rstrip("/"),
        "resource": ANALYZER_API_RESOURCE,
    },
    "anonymizer": {
        "url": os.environ.get(
            "ANONYMIZER_BASE_URL", "https://presidio-anonymizer.ase-jtl.appserviceenvironment.us"
        ).rstrip("/"),
        "resource": ANONYMIZER_API_RESOURCE,
    },
}
LANGUAGE = "en"
TIMEOUT = (10, 60)
BATCH_SIZE = 1  # Increase only when the deployed analyzer accepts text arrays.
ANONYMIZERS = {"DEFAULT": {"type": "replace", "new_value": "[REDACTED]"}}


## Connect to both services

Use each API's registered Application ID URI (for example `api://<API-app-registration-client-id>`), not the VM identity's client ID, a guessed hostname, or the Resource Manager audience. Do not append `/.default`; the request helper adds it.

`BATCH_SIZE = 1` uses scalar `/analyze` requests for compatibility with older deployed images. Set a larger positive integer to send arrays of texts to a batch-capable analyzer's **same `/analyze` endpoint**. This is not an automatic fallback: incompatible images or unexpected batch responses produce an explicit error. `/anonymize` remains one request per detected text. Requests are sequential and bounded by batch size; there are no client worker processes. Choose a size consistent with your services' request-size limits and load budget.


In [ ]:
def validate_services() -> None:
    """Validate API configuration before making requests.

    :return: None.
    :raises ValueError: Invalid URL, API resource, or batch size.
    """
    if type(BATCH_SIZE) is not int or BATCH_SIZE < 1:
        raise ValueError("BATCH_SIZE must be a positive integer.")
    for name, service in SERVICES.items():
        try:
            parsed = urlsplit(service["url"])
            port = parsed.port
        except ValueError:
            raise ValueError(f"Invalid {name} HTTPS base URL.") from None
        if (
            parsed.scheme != "https" or not parsed.hostname
            or parsed.username is not None or parsed.password is not None
            or parsed.path or parsed.query or parsed.fragment
            or port not in (None, 443)
        ):
            raise ValueError(f"Configure a root HTTPS base URL for {name}.")
        if not service["resource"] or service["resource"].endswith("/.default"):
            raise ValueError(f"Set {name.upper()}_API_RESOURCE to its registered URI without /.default.")


validate_services()
credential = (
    ManagedIdentityCredential(client_id=MANAGED_IDENTITY_CLIENT_ID)
    if MANAGED_IDENTITY_CLIENT_ID
    else ManagedIdentityCredential()
)
session = requests.Session()
# Prevent .netrc credentials from replacing the bearer token.
session.auth = lambda request: request


def api_request(name: str, path: str, payload: dict | None = None) -> requests.Response:
    """Call a service without exposing tokens or error bodies.

    :param name: Analyzer or anonymizer service name.
    :param path: Endpoint path.
    :param payload: JSON body, or None for GET.
    :return: Successful HTTP response.
    :raises RuntimeError: Token, transport, or HTTP failure.
    """
    service = SERVICES[name]
    try:
        token = credential.get_token(service["resource"].rstrip("/") + "/.default")
    except (AzureError, ValueError):
        raise RuntimeError(f"Token acquisition failed for {name}; check the VM identity and API resource.") from None
    try:
        response = session.request(
            "GET" if payload is None else "POST", service["url"] + path,
            headers={"Authorization": "Bearer " + token.token},
            json=payload, timeout=TIMEOUT, allow_redirects=False, verify=True,
        )
    except requests.exceptions.SSLError:
        raise RuntimeError(f"TLS validation failed for {name}; check the trusted CA chain.") from None
    except requests.exceptions.Timeout:
        raise RuntimeError(f"{name} request timed out; check connectivity and service health.") from None
    except requests.exceptions.RequestException:
        raise RuntimeError(f"{name} request failed; check private DNS and HTTPS connectivity.") from None
    if response.status_code != 200:
        raise RuntimeError(f"{name} returned HTTP {response.status_code}; response body omitted for privacy.")
    return response


def response_json(response: requests.Response, name: str):
    """Decode JSON without exposing a malformed response.

    :param response: HTTP response.
    :param name: Service name for the error message.
    :return: Decoded JSON value.
    :raises RuntimeError: Response is not valid JSON.
    """
    try:
        return response.json()
    except ValueError:
        raise RuntimeError(f"{name} returned invalid JSON; check the endpoint and image version.") from None


for name in SERVICES:
    api_request(name, "/health")
    print(f"{name}: health endpoint returned HTTP 200")


## Structure-preserving REST helpers

Strings are analyzed without stripping or normalization, and the exact same text and validated offsets are sent to the anonymizer. Detected values are replaced with `[REDACTED]`. Empty strings and texts with no detections are unchanged; no-detection results are not a privacy certification.

Integers, floats, and booleans are analyzed using their string representation, as in the local batch sample. If there are no detections, their original types are preserved instead of coercing every primitive-list value to a string. If a numeric or boolean value is detected as PII, its anonymized result is a **string**. Nulls and empty strings remain unchanged. Store identifiers as strings when leading zeros or formatting must be retained. Non-finite floats, sets, custom objects, and non-string dictionary keys are rejected; use JSON-compatible inputs. Dictionary keys are used as detection context, not anonymized themselves, and may also contain PII.

Nested dictionaries and lists (including lists of objects) are supported. Homogeneous primitive lists are processed in chunks; mixed/nested lists are traversed recursively. Bare skip names match at any depth; dotted paths match the full dictionary-key path, including inside list elements. Skipping a container skips its entire subtree. Skipped values remain sensitive. The input is not modified, and failures raise rather than returning a partially processed output. Large datasets should be supplied as separately processed partitions instead of loading them all into one DataFrame.


In [ ]:
def validate_detections(data, text: str) -> list[dict]:
    """Validate and minimize detections forwarded to the anonymizer.

    :param data: Analyzer JSON result for one text.
    :param text: Original analyzed text.
    :return: Detection dictionaries containing only required fields.
    :raises RuntimeError: Invalid response shape, offsets, score, or entity.
    """
    if not isinstance(data, list):
        raise RuntimeError("Analyzer response must be a list of detections.")
    validated = []
    for result in data:
        if not isinstance(result, dict):
            raise RuntimeError("Each analyzer detection must be an object.")
        start, end = result.get("start"), result.get("end")
        score, entity = result.get("score"), result.get("entity_type")
        if (
            type(start) is not int or type(end) is not int
            or not 0 <= start < end <= len(text)
            or type(score) not in (int, float) or not math.isfinite(score)
            or not 0 <= score <= 1
            or not isinstance(entity, str) or not entity.strip()
        ):
            raise RuntimeError("Analyzer returned invalid offsets, score, or entity type.")
        validated.append({"start": start, "end": end, "score": score, "entity_type": entity})
    return validated


def validate_data(value) -> None:
    """Reject unsupported structures before any API calls.

    :param value: JSON-compatible data to process.
    :return: None.
    :raises ValueError: Unsupported value or non-string dictionary key.
    """
    if isinstance(value, dict):
        for key, child in value.items():
            if not isinstance(key, str):
                raise ValueError("Dictionary keys must be strings.")
            validate_data(child)
    elif isinstance(value, list):
        for child in value:
            validate_data(child)
    elif value is None or type(value) in (str, int, bool):
        return
    elif type(value) is float and math.isfinite(value):
        return
    else:
        raise ValueError("Use JSON values only: dict, list, string, finite number, boolean, or null.")


def anonymize_values(values: list, context: list[str] | None = None) -> list:
    """Process a primitive list in bounded chunks, preserving order and types.

    :param values: List of JSON primitive values.
    :param context: Dictionary key context for detection.
    :return: New list; detected primitives become strings, others keep their types.
    :raises ValueError: Invalid input or batch size.
    :raises RuntimeError: Service failure or malformed/misaligned response.
    """
    if not isinstance(values, list) or any(isinstance(v, (dict, list)) for v in values):
        raise ValueError("anonymize_values expects a list of JSON primitives.")
    validate_data(values)
    if type(BATCH_SIZE) is not int or BATCH_SIZE < 1:
        raise ValueError("BATCH_SIZE must be a positive integer.")
    output = list(values)
    for offset in range(0, len(values), BATCH_SIZE):
        positions = [
            index for index in range(offset, min(offset + BATCH_SIZE, len(values)))
            if values[index] is not None and values[index] != ""
        ]
        if not positions:
            continue
        texts = [str(values[index]) for index in positions]
        payload = {"text": texts[0] if BATCH_SIZE == 1 else texts, "language": LANGUAGE}
        if context:
            payload["context"] = context
        data = response_json(api_request("analyzer", "/analyze", payload), "analyzer")
        batches = [data] if BATCH_SIZE == 1 else data
        if not isinstance(batches, list) or len(batches) != len(texts):
            raise RuntimeError("Analyzer batch response count does not match the submitted texts.")
        detections = [validate_detections(result, text) for result, text in zip(batches, texts)]
        for index, text, results in zip(positions, texts, detections):
            if not results:
                continue
            result = response_json(api_request("anonymizer", "/anonymize", {
                "text": text, "analyzer_results": results, "anonymizers": ANONYMIZERS,
            }), "anonymizer")
            if not isinstance(result, dict) or not isinstance(result.get("text"), str):
                raise RuntimeError("Anonymizer response must contain a string text field.")
            output[index] = result["text"]
    return output


def anonymize_data(value, keys_to_skip: list[str] | None = None):
    """Anonymize JSON values without mutating inputs or skipped subtrees.

    :param value: JSON-compatible scalar, list, or nested dictionary.
    :param keys_to_skip: Bare key names or dotted dictionary-key paths.
    :return: Structure-preserving anonymized data.
    :raises ValueError: Invalid input or skip specification.
    :raises RuntimeError: API failure or malformed response.
    """
    validate_data(value)
    skips = [] if keys_to_skip is None else keys_to_skip
    if not isinstance(skips, list) or any(
        not isinstance(key, str) or not key or any(not part for part in key.split("."))
        for key in skips
    ):
        raise ValueError("keys_to_skip must be a list of nonempty key names or dotted paths.")
    names = {key for key in skips if "." not in key}
    paths = {tuple(key.split(".")) for key in skips if "." in key}

    def walk(current, path):
        if isinstance(current, dict):
            output = {}
            for key, child in current.items():
                child_path = path + (key,)
                output[key] = (
                    copy.deepcopy(child) if key in names or child_path in paths
                    else walk(child, child_path)
                )
            return output
        if isinstance(current, list):
            if any(isinstance(child, (dict, list)) for child in current):
                return [walk(child, path) for child in current]
            return anonymize_values(current, context=list(path))
        if current is not None:
            return anonymize_values([current], context=list(path))[0]
        return current

    return walk(value, ())


## Example: Pandas table

Convert columns to lists, process primitive cells, then restore the original column order and index. Only the shape is displayed. For real tables, normalize Pandas missing-value sentinels to JSON nulls and scalar types to native Python types before calling the helpers. Columns containing detected numeric PII may change dtype because replacement values are strings.


In [ ]:
columns = ["name phrase", "phone number phrase", "integer", "boolean"]
sample_data = [
    ("Charlie likes this", "Please call 202-555-0101 after 2pm", 1, True),
    ("You should talk to Mike", "His number is 202-555-0102", 2, False),
    ("Mary had a little startup", "Phone number: 202-555-0103", 3, False),
]
df = pd.DataFrame(sample_data, columns=columns)
anonymized_columns = anonymize_data(df.to_dict(orient="list"))
scrubbed_df = pd.DataFrame(anonymized_columns, columns=df.columns, index=df.index)
assert scrubbed_df["integer"].tolist() == df["integer"].tolist()
assert scrubbed_df["boolean"].tolist() == df["boolean"].tolist()
print(f"Processed table: {scrubbed_df.shape[0]} rows, {scrubbed_df.shape[1]} columns.")


## Example: nested JSON and lists of objects

The URL collection is a JSON list, not the Python set in the original sample. No source or output dictionaries are printed. For your own JSON file, load it into a Python value on the kernel host and pass that value to `anonymize_data`; protect both the original and returned data.


In [ ]:
nested_dict = {
    "key_a": {"key_a1": "My phone number is 202-555-0104"},
    "key_b": ["https://example.test"],
    "key_c": 3,
    "names": ["James Bond", "Clark Kent", "No name here!"],
    "contacts": [{"email": "alpha@example.test"}, {"email": "beta@example.test"}],
}
anonymized_json = anonymize_data(nested_dict)
print(f"Processed JSON with {len(anonymized_json)} top-level keys.")


## Intentionally skip fields

Bare names such as `key_a1` match at any depth. `key_a.key_a1` skips only that complete path; similarly prefixed siblings do not match. Paths through arrays omit list indices, so `contacts.email` skips email values in every contact object. A dot in a skip specification means a path separator; use a bare name without dots or restructure literal dotted keys if they need skipping.

**These outputs deliberately retain sensitive values. Do not display them as proof of anonymization.**


In [ ]:
skipped_by_name = anonymize_data(nested_dict, keys_to_skip=["key_a1", "names"])
assert skipped_by_name["key_a"] == nested_dict["key_a"]
assert skipped_by_name["names"] == nested_dict["names"]

skipped_by_path = anonymize_data(nested_dict, keys_to_skip=["key_a.key_a1", "contacts.email"])
assert skipped_by_path["key_a"]["key_a1"] == nested_dict["key_a"]["key_a1"]
assert skipped_by_path["contacts"] == nested_dict["contacts"]
print("Processed skip examples; deliberately skipped fields were preserved.")


## Example: bounded list processing

This replaces the local spaCy multiprocessing/process-monitor example. Start with a small list; increase workload and `BATCH_SIZE` only after confirming your deployed analyzer's array support and service capacity. A batch-size setting controls request grouping, not server worker counts. Earlier successful requests are not rolled back if a later request fails, but no partial output is returned.


In [ ]:
list_of_texts = [f"Email person{index}@example.test" for index in range(4)]
anonymized_list = anonymize_values(list_of_texts)
assert len(anonymized_list) == len(list_of_texts)
print(f"Processed {len(anonymized_list)} list items with BATCH_SIZE={BATCH_SIZE}.")


## Troubleshooting and cleanup

| Failure | Check |
| --- | --- |
| Blank API resource | Edit both resource assignments in the configuration cell or set their environment variables on the kernel host; rerun configuration before the client cell. |
| Token acquisition | VM identity endpoint, attached user-assigned identity and its client ID, registered API resource, and required app roles. |
| HTTP 401/403 | API audience, issuer/token version, caller client ID and principal ID allowlists, tenant restrictions, and App Service network rules. |
| TLS/timeout/connection failure | Private DNS, routing, HTTPS port 443, proxy configuration, and trusted CA chain. Do not disable TLS verification. |
| Batch shape failure or HTTP 400/500 after increasing batch size | Confirm deployed `/analyze` supports text arrays. Choose `BATCH_SIZE = 1` explicitly and rerun; there is no silent retry/fallback. |
| HTTP 413 | Reduce batch size/text size to fit the deployed services' limits. |
| Unsupported input | Normalize Pandas nulls and scalar types to JSON-compatible values; use lists instead of sets. |

From the repository root, maintainers can run `python -m unittest discover -s docs/samples/ASE/tests -p test_batch_processing.py -v`. Tests use mocked APIs and synthetic data, not a live ASE.

Close the session and credential when finished. Rerun the client cell before sending further requests.


In [ ]:
session.close()
credential.close()
